# Libras Livre — treino em GPU (Colab ou Kaggle)

Roda o mesmo código de `computer-vision-model/treino/` numa GPU gratuita. O que muda
em relação ao notebook local é só a velocidade: no CPU cada rodada da
leave-one-signer-out leva ~20 min; na GPU, minutos.

**O que NÃO se move para cá:** a extração de landmarks. Ela é MediaPipe em CPU, dura
horas e GPU não acelera — continua rodando na máquina local.

---

## Antes de começar

1. **Ative a GPU:** Colab → *Ambiente de execução > Alterar tipo de ambiente > GPU*.
   Kaggle → painel direito, *Accelerator > GPU*.
2. **Tenha o `landmarks-minds.tar.gz`** à mão (41 MB, 800 clipes do MINDS). Na máquina
   local ele fica em `~/landmarks-minds.tar.gz`; para regerar:
   ```bash
   cd computer-vision-model/PoC/data
   mkdir -p /tmp/pack/landmarks && cp landmarks/pessoaM*.npy /tmp/pack/landmarks/
   tar czf ~/landmarks-minds.tar.gz -C /tmp/pack landmarks
   ```
   Só MINDS de propósito: é licença MIT, e o treino usa `--fontes minds` de qualquer
   forma. Os 30 clipes da V-LIBRASIL são CC BY-NC-**ND** e não devem circular.

## 1. Ambiente e código

In [ ]:
import os, pathlib, shutil, subprocess, sys

EM_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
EM_KAGGLE = os.path.exists("/kaggle/working")
BASE = pathlib.Path("/content" if EM_COLAB else "/kaggle/working" if EM_KAGGLE else ".")
print("ambiente:", "Colab" if EM_COLAB else "Kaggle" if EM_KAGGLE else "local", "| base:", BASE)

URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
BRANCH = "claude/libras-detection-model-53kd30"   # onde vive o código de modelo
REPO = BASE / "libras-livre-ai-glasses-brasil"
TREINO = REPO / "computer-vision-model" / "treino"

def branch_atual(repo):
    r = subprocess.run(["git", "-C", str(repo), "rev-parse", "--abbrev-ref", "HEAD"],
                       capture_output=True, text=True)
    return r.stdout.strip()

# Idempotente de verdade: um clone que já existe MAS está na branch errada (ou
# incompleto, de uma tentativa anterior) é descartado e refeito. Só checar se o
# diretório existe deixaria um clone ruim para trás, e o erro apareceria depois,
# longe da causa.
if REPO.exists() and (branch_atual(REPO) != BRANCH or not TREINO.is_dir()):
    print(f"clone existente está em '{branch_atual(REPO)}' — refazendo")
    shutil.rmtree(REPO)

if not REPO.exists():
    # --branch no clone: sem isso vem a branch default (main), que não tem treino/.
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, str(REPO)],
                   check=True)

assert TREINO.is_dir(), f"{TREINO} não existe mesmo após o clone"
print("branch:", branch_atual(REPO))
print("código em:", TREINO)
print("arquivos:", sorted(p.name for p in TREINO.glob("*.py")))

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! sem GPU — ative o acelerador (Ambiente de execução > Alterar tipo) "
          "antes de treinar, senão isto vira CPU lento")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=False)

## 2. Landmarks

O destino é `computer-vision-model/PoC/data/landmarks/`, que é onde `treinar.py`
procura (o caminho vem de `PoC/config.yaml`).

In [ ]:
import tarfile

DESTINO = REPO / "computer-vision-model" / "PoC" / "data"
DESTINO.mkdir(parents=True, exist_ok=True)

if EM_COLAB:
    from google.colab import files
    print("selecione o landmarks-minds.tar.gz")
    enviados = files.upload()
    origem = pathlib.Path(next(iter(enviados)))
else:
    # Kaggle: adicione o .tar.gz como Dataset e ajuste o caminho.
    origem = pathlib.Path("/kaggle/input/libras-landmarks/landmarks-minds.tar.gz")

# Alternativa no Colab: montar o Drive e apontar para o arquivo lá
# from google.colab import drive; drive.mount('/content/drive')
# origem = pathlib.Path('/content/drive/MyDrive/libras/landmarks-minds.tar.gz')

assert origem.exists(), f"não achei {origem}"
with tarfile.open(origem) as tar:
    try:
        tar.extractall(DESTINO, filter="data")   # filter= exigido no Python 3.12+
    except TypeError:
        tar.extractall(DESTINO)                   # versões anteriores

npys = sorted((DESTINO / "landmarks").glob("*.npy"))
pessoas = sorted({p.name.split("_")[0] for p in npys})
print(f"{len(npys)} arquivos de landmarks | {len(pessoas)} pessoas: {', '.join(pessoas)}")
assert npys, "nada extraído — confira o arquivo enviado"

## 3. Treino

`--dispositivo auto` pega a GPU sozinho. As duas arquiteturas usam exatamente o mesmo
protocolo (leave-one-signer-out, uma pessoa inteira fora por rodada), então os números
são comparáveis entre si e com o baseline DTW da PoC (70,0%) e a ResNet local (93,4%).

In [ ]:
# os.chdir em vez do %cd: a substituição {VAR} do IPython não avisa quando falha
os.chdir(TREINO)
print("cwd:", os.getcwd())

# Validação do encanamento com dados sintéticos (segundos). Se falhar, pare aqui.
!python selftest.py

In [ ]:
# ST-GCN — grafo do esqueleto, 0,46M parâmetros. É ESTE o comparativo.
!python treinar.py --arquitetura gcn --dispositivo auto --batch 64

In [ ]:
# Opcional: reproduzir a ResNet-18 na mesma GPU, para comparar sem viés de hardware
!python treinar.py --arquitetura resnet --dispositivo auto --batch 64

## 4. Resultados

**Baixe os relatórios antes de fechar a sessão** — o disco do Colab é descartado ao
encerrar.

In [ ]:
for arq in ("gcn", "resnet"):
    rel = TREINO / f"resultados-{arq}" / "relatorio.md"
    if rel.exists():
        print("=" * 70)
        print(rel.read_text(encoding="utf-8")[:1500])

if EM_COLAB:
    from google.colab import files
    for arq in ("gcn", "resnet"):
        rel = TREINO / f"resultados-{arq}" / "relatorio.md"
        if rel.exists():
            files.download(str(rel))